# Feature Store Verification

Verify that Feature Store sync completed and online store is serving data.

**Prerequisites:**
1. `python -m feature_store.ingest` — populate BQ feature table
2. `python -m feature_store.setup` — create online store + feature view
3. `python -m feature_store.sync` — sync BQ → online store

## Config

In [1]:
PROJECT = "deeplearning-sahil"
REGION = "us-central1"
ONLINE_STORE_ID = "ml_online_store"
FEATURE_VIEW_ID = "iris_features"

API_ENDPOINT = f"{REGION}-aiplatform.googleapis.com"
FEATURE_VIEW_PATH = (
    f"projects/{PROJECT}/locations/{REGION}"
    f"/featureOnlineStores/{ONLINE_STORE_ID}"
    f"/featureViews/{FEATURE_VIEW_ID}"
)

## 1. Check Sync Status

List recent syncs and verify the latest one succeeded.

In [2]:
from google.cloud.aiplatform_v1 import FeatureOnlineStoreAdminServiceClient

admin_client = FeatureOnlineStoreAdminServiceClient(
    client_options={"api_endpoint": API_ENDPOINT}
)

syncs = list(admin_client.list_feature_view_syncs(parent=FEATURE_VIEW_PATH))
print(f"Total syncs: {len(syncs)}\n")

for sync in syncs[:5]:
    print(f"Name: {sync.name}")
    print(f"  Run time:     {sync.run_time}")
    print(f"  Final status: {sync.final_status}")
    print()

Total syncs: 1

Name: projects/deeplearning-sahil/locations/us-central1/featureOnlineStores/ml_online_store/featureViews/iris_features/featureViewSyncs/5288896071303430144
  Run time:     start_time {
  seconds: 1781611064
  nanos: 37817000
}
end_time {
  seconds: 1781611100
  nanos: 899333000
}

  Final status: 



## 2. Read Features from Online Store

Fetch feature values for a specific entity to confirm the online store is serving.

In [3]:
from google.cloud.aiplatform_v1 import (
    FeatureOnlineStoreServiceClient,
    FetchFeatureValuesRequest,
    FeatureViewDataKey,
)

serving_client = FeatureOnlineStoreServiceClient(
    client_options={"api_endpoint": API_ENDPOINT}
)

In [4]:
# Fetch a single training entity
entity_id = "1_training"

response = serving_client.fetch_feature_values(
    feature_view=FEATURE_VIEW_PATH,
    data_key=FeatureViewDataKey(key=entity_id),
)

print(f"Entity: {entity_id}")
print(f"Features:\n{response.key_values}")

Entity: 1_training
Features:
features {
  value {
    double_value: 5.1
  }
  name: "sepal_length_cm"
}
features {
  value {
    double_value: 3.5
  }
  name: "sepal_width_cm"
}
features {
  value {
    double_value: 1.4
  }
  name: "petal_length_cm"
}
features {
  value {
    double_value: 0.2
  }
  name: "petal_width_cm"
}
features {
  value {
    string_value: "Iris-setosa"
  }
  name: "species"
}
features {
  value {
    string_value: "training"
  }
  name: "source"
}
features {
  value {
    int64_value: 1781566511217692
  }
  name: "feature_timestamp"
}



In [6]:
# Fetch a batch_input entity (if you ran bq_dataloader --generate-random)
entity_id = "1_batch_input"

response = serving_client.fetch_feature_values(
    feature_view=FEATURE_VIEW_PATH,
    data_key=FeatureViewDataKey(key=entity_id),
)

print(f"Entity: {entity_id}")
print(f"Features:\n{response.key_values}")

Entity: 1_batch_input
Features:
features {
  value {
    double_value: 7.7
  }
  name: "sepal_length_cm"
}
features {
  value {
    double_value: 3.1
  }
  name: "sepal_width_cm"
}
features {
  value {
    double_value: 1.6
  }
  name: "petal_length_cm"
}
features {
  value {
    double_value: 2.1
  }
  name: "petal_width_cm"
}
features {
  name: "species"
}
features {
  value {
    string_value: "batch_input"
  }
  name: "source"
}
features {
  value {
    int64_value: 1781566559500473
  }
  name: "feature_timestamp"
}



## 3. Read Streaming Features from Online Store

Fetch feature values written by the Dataflow streaming pipeline (`iris_feature_pipeline.py`).
Streaming entities have entity_id format: `{sample_id}_streaming`.

In [ ]:
# Fetch a streaming entity written by the Dataflow feature pipeline
entity_id = "1_streaming"

response = serving_client.fetch_feature_values(
    feature_view=FEATURE_VIEW_PATH,
    data_key=FeatureViewDataKey(key=entity_id),
)

print(f"Entity: {entity_id}")
print(f"Features:\n{response.key_values}")